# 02 — Utilization & Downtime Analysis

Healthcare Imaging Equipment Utilization & Downtime Analysis

**Month 2-3 milestone (Feb-Mar 2023):** compute utilization rate per machine, identify the top causes of downtime, and summarize per-machine reliability.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'notebooks'))
import pandas as pd
import data_utils as du

equipment_logs_clean = pd.read_csv('../outputs/exports/equipment_logs_clean.csv', parse_dates=['date', 'install_date'])
maintenance_tickets_clean = pd.read_csv('../outputs/exports/maintenance_tickets_clean.csv', parse_dates=['reported_date', 'resolved_date'])
machine_metadata_clean = pd.read_csv('../outputs/exports/machine_metadata_clean.csv', parse_dates=['last_service_date'])

## Utilization rate

`utilization_rate = actual usage hours / available hours`, where available hours assumes a 24-hour/day operating window per machine over the days it was observed in service.

In [2]:
utilization = du.compute_utilization(equipment_logs_clean)
utilization.sort_values('utilization_rate', ascending=False)

,machine_id,machine_type,hospital_site,total_usage_hours,days_observed,available_hours,utilization_rate
8,XR-303,X-ray,St. Anne's Hospital,4333.200,365,8760.0,0.4947
7,XR-302,X-ray,Lakeview Medical Center,4308.860,365,8760.0,0.4919
6,XR-301,X-ray,Riverside General,4277.010,365,8760.0,0.4882
2,CT-203,CT,St. Anne's Hospital,3372.515,363,8712.0,0.3871
0,CT-201,CT,Riverside General,3337.420,365,8760.0,0.3810
1,CT-202,CT,Lakeview Medical Center,3321.880,365,8760.0,0.3792
5,MRI-103,MRI,St. Anne's Hospital,2795.390,365,8760.0,0.3191
4,MRI-102,MRI,Lakeview Medical Center,2732.815,365,8760.0,0.3120
3,MRI-101,MRI,Riverside General,2730.470,365,8760.0,0.3117


## Downtime per machine (total hours, ticket count, MTTR)

In [3]:
downtime_summary = du.downtime_per_machine(maintenance_tickets_clean)
downtime_summary

,machine_id,total_downtime_hours,ticket_count,mttr_hours
1,CT-202,108.4,14,7.74
3,MRI-101,96.0,9,10.66
6,XR-301,86.4,10,8.64
7,XR-302,81.8,6,13.64
0,CT-201,73.8,8,9.22
8,XR-303,38.0,4,9.50
2,CT-203,29.6,6,4.92
4,MRI-102,11.9,2,5.94
5,MRI-103,9.2,2,4.61


## Top 3 causes of downtime fleet-wide

In [4]:
top_causes = du.top_downtime_causes(maintenance_tickets_clean, n=3)
top_causes

,issue_type,occurrences,total_downtime_hours
0,Calibration drift,15,137.5
4,Generator fault,6,86.0
3,Detector fault,9,73.4


## Utilization vs. downtime, joined with machine age

Sets up the table used by the R regression script (`age_downtime_regression.R`) and by the visualization step.

In [5]:
combined = (
    utilization
    .merge(downtime_summary, on='machine_id', how='left')
    .merge(machine_metadata_clean[['machine_id', 'age_years']], on='machine_id', how='left')
)
combined['total_downtime_hours'] = combined['total_downtime_hours'].fillna(0)
combined['ticket_count'] = combined['ticket_count'].fillna(0)
combined.to_csv('../outputs/exports/utilization_downtime_by_machine.csv', index=False)
utilization.to_csv('../outputs/exports/utilization_summary.csv', index=False)
downtime_summary.to_csv('../outputs/exports/downtime_summary.csv', index=False)
top_causes.to_csv('../outputs/exports/top_downtime_causes.csv', index=False)
combined

,machine_id,machine_type,hospital_site,total_usage_hours,days_observed,available_hours,utilization_rate,total_downtime_hours,ticket_count,mttr_hours,age_years
0,CT-201,CT,Riverside General,3337.420,365,8760.0,0.3810,73.8,8,9.22,7.57
1,CT-202,CT,Lakeview Medical Center,3321.880,365,8760.0,0.3792,108.4,14,7.74,5.12
2,CT-203,CT,St. Anne's Hospital,3372.515,363,8712.0,0.3871,29.6,6,4.92,0.99
3,MRI-101,MRI,Riverside General,2730.470,365,8760.0,0.3117,96.0,9,10.66,7.05
4,MRI-102,MRI,Lakeview Medical Center,2732.815,365,8760.0,0.3120,11.9,2,5.94,3.75
5,MRI-103,MRI,St. Anne's Hospital,2795.390,365,8760.0,0.3191,9.2,2,4.61,1.36
6,XR-301,X-ray,Riverside General,4277.010,365,8760.0,0.4882,86.4,10,8.64,8.78
7,XR-302,X-ray,Lakeview Medical Center,4308.860,365,8760.0,0.4919,81.8,6,13.64,5.47
8,XR-303,X-ray,St. Anne's Hospital,4333.200,365,8760.0,0.4947,38.0,4,9.50,3.17


In [6]:
# Aggregate by hospital site for the utilization heatmap in the visualization step
site_daily = (
    equipment_logs_clean
    .assign(month=equipment_logs_clean['date'].dt.to_period('M').astype(str))
    .groupby(['hospital_site', 'month'])['daily_usage_hours']
    .sum()
    .reset_index()
)
site_daily.to_csv('../outputs/exports/utilization_by_site_month.csv', index=False)

downtime_trend = (
    maintenance_tickets_clean
    .merge(equipment_logs_clean[['machine_id', 'machine_type']].drop_duplicates(), on='machine_id', how='left')
    .assign(month=maintenance_tickets_clean['reported_date'].dt.to_period('M').astype(str))
    .groupby(['machine_type', 'month'])['downtime_hours']
    .sum()
    .reset_index()
)
downtime_trend.to_csv('../outputs/exports/downtime_trend_by_type_month.csv', index=False)
print('Chart-ready aggregates written to outputs/exports/')

Chart-ready aggregates written to outputs/exports/
